# Solution visualisation

Solves an instance with the shared package and draws each crew member's duty sequence. Default: the briefing's ABC example, then the synthetic `medium` preset (**synthetic data**).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from railway_rostering import load_instance, solve_instance, ModelConfig, SolverConfig
from railway_rostering.reporting import assignment_rows, uncovered_rows, console_summary

def gantt(result, max_crew=40, title=""):
    rows = pd.DataFrame(assignment_rows(result))
    if rows.empty:
        print("no assignments"); return
    rows["start"] = pd.to_datetime(rows["start"]); rows["end"] = pd.to_datetime(rows["end"])
    crew_ids = sorted(rows["crew_id"].unique())[:max_crew]
    acts = sorted(rows["activity"].unique())
    colors = {a: plt.cm.tab10(i % 10) for i, a in enumerate(acts)}
    fig, ax = plt.subplots(figsize=(13, 0.35 * len(crew_ids) + 1.5))
    for y, cid in enumerate(crew_ids):
        for _, r in rows[rows["crew_id"] == cid].iterrows():
            ax.barh(y, (r["end"] - r["start"]).total_seconds() / 86400, left=r["start"], color=colors[r["activity"]],
                    edgecolor="black", hatch="//" if r["assignment_type"] == "existing" else None)
            ax.text(r["start"], y, f" {r['start_location'][:3]}>{r['end_location'][:3]}", va="center", fontsize=7)
    ax.set_yticks(range(len(crew_ids))); ax.set_yticklabels(crew_ids); ax.invert_yaxis()
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=acts, fontsize=7,
              loc="upper left", bbox_to_anchor=(1, 1))
    ax.set_title(title + "  (hatched = existing assignment)"); plt.tight_layout()

## Briefing ABC example

Three 5 PM (night) duties, a rest/transition day on 4 Nov, then three 5 AM (day) duties. The model must leave >= 23 h between the last night duty and the first day duty, and ABC must be at base Howrah in between.

In [ ]:
abc = solve_instance(load_instance(ROOT / "data/sample/briefing_abc"), solver_config=SolverConfig(num_workers=2))
print(console_summary(abc))
gantt(abc, title="briefing_abc")
display(pd.DataFrame(assignment_rows(abc))[["sequence", "duty_id", "activity", "start", "end", "start_location", "end_location", "shift_type", "start_offset_minutes"]])

## Synthetic `medium` preset

In [ ]:
med = solve_instance(load_instance(ROOT / "data/generated/medium"), solver_config=SolverConfig(time_limit_seconds=30, num_workers=2))
print(console_summary(med))
gantt(med, max_crew=35, title="medium (first 35 working crew)")

## Why are duties uncovered?

In [ ]:
unc = pd.DataFrame(uncovered_rows(med))
if not unc.empty:
    unc["reason_code"].value_counts().plot.barh(title="Uncovered duties by reason")
    display(unc[["duty_id", "activity", "start", "start_location", "uncovered_units", "reason_code", "n_eligible", "n_available", "n_time_ok", "n_feasible_candidates"]])

## Lift-up / layback usage (Excess) and time changes

In [ ]:
rows = pd.DataFrame(assignment_rows(med))
display(pd.crosstab(rows["crew_type"], rows["start_offset_minutes"]))